# C.R.A.T.E.R — Localize (pixel → map)

**Input:** detections from the vision teammate (class, confidence, bounding box in pixels, frame id) + the rover pose per frame.
**Output:** a short list of *telemetry events*: one stable hazard, with a grid cell, an uncertainty, and a timestamp.

Pipeline: `detection → ground pixel → ray → ground point (rover frame) → world frame → grid cell → uncertainty → merge repeats → events`

**How to use this notebook**
1. Put `detections.jsonl` (from the perception folder) next to this notebook.
2. The camera numbers in the `CFG` cell are **assumptions** (the dataset page gives none). Replace them if you ever get the real ones.
3. Run all. Section 11 shows how much the results change if those assumptions are wrong. The last cell writes `telemetry_events.json` for the telemetry step.
4. To run the built-in fake-data test instead, set `USE_SYNTHETIC = True` in the *Load data* cell.

## 1. Setup and assumptions

In [1]:
import json, math
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
CFG = dict(
    # --- camera: REPLACE with the real values from the dataset / simulator ---
    # ASSUMPTIONS: the dataset page gives no camera numbers, and no horizon is visible in the frames
    # (so the camera looks steeply down). Replace these if you get the real values.
    img_w=640, img_h=640,        # image size in pixels (dataset images are 640x640)
    hfov_deg=60.0,               # horizontal field of view
    cam_height_m=0.5,            # camera height above the ground
    cam_pitch_deg=40.0,          # how far the camera looks DOWN from horizontal

    # --- limits ---
    min_range_m=0.1,
    max_range_m=25.0,            # beyond this the flat-ground estimate is too unreliable

    # --- error model ---
    pix_sigma=3.0,               # assumed detector error, in pixels
    pose_sigma0_m=0.05,          # rover position error at start
    pose_drift_per_m=0.01,       # extra position error per metre travelled
    sigma_floor_m=0.30,          # merging never claims better than this (pose drift is shared by all sightings, averaging can't remove it)

    # --- grid and merging ---
    cell_size_m=0.25,
    gate_m=0.5,                  # closer than this to an existing hazard (+ sigma) = same hazard
    min_conf=0.4,                # ignore weak detections
    min_obs=3,                   # sightings needed before a hazard is confirmed
)

# Camera used ONLY by the fake-data test (a far-looking camera, hazards 2 to 20 m away)
SYN_CFG = dict(CFG, hfov_deg=90.0, cam_height_m=0.6, cam_pitch_deg=15.0,
               min_range_m=0.5, cell_size_m=1.0, gate_m=1.5)

## 2. Input contract (what the teammate must give us)

**Detections**: one record per detected object per frame:

```json
{"frame_id": 12, "timestamp": 1.2, "class": "rock", "confidence": 0.87, "bbox": [x1, y1, x2, y2]}
```
`bbox` is in pixels, origin top-left, x to the right, y downwards. `x1,y1,x2,y2` as separate keys also works.

**JSONL files** (one line per frame, detections inside a `dets` list, as in the perception folder) are read directly by `flatten_frames`.

**Poses** (rover position per frame, in metres / radians, world frame with the landing site at (0, 0)):
`frame_id, x, y, yaw`  (yaw = 0 means facing +x, positive = counter-clockwise).

If the teammate's key names differ, edit `FIELD_MAP` below instead of touching the rest of the code.

In [3]:
# our name  ->  teammate's name   (edit the right-hand side)
FIELD_MAP = dict(frame_id="frame_id", timestamp="timestamp", cls="class",
                 confidence="confidence", bbox="bbox")

def normalize_detection(raw, fm=FIELD_MAP):
    """Convert one raw record into our standard format."""
    if fm["bbox"] in raw:
        bbox = [float(v) for v in raw[fm["bbox"]]]
    else:
        bbox = [float(raw[k]) for k in ("x1", "y1", "x2", "y2")]
    fid = int(raw[fm["frame_id"]])
    return dict(frame_id=fid,
                timestamp=float(raw.get(fm["timestamp"], fid)),
                cls=str(raw[fm["cls"]]),
                confidence=float(raw[fm["confidence"]]),
                bbox=bbox)

def flatten_frames(path):
    """JSONL = one JSON object per FRAME, with its detections in a 'dets' list. Returns one record per detection."""
    out = []
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        try:
            fr = json.loads(line)
        except json.JSONDecodeError:
            continue                      # skip a broken / truncated line
        for d in fr.get("dets", []):
            out.append({"frame_id": fr["frame"], "timestamp": fr["t"], "class": d["name"],
                        "confidence": d["conf"], "bbox": d["bbox"], "track_id": d.get("track_id")})
    return out

def load_detections(src):
    """src: path to .jsonl / .json / .csv, or an in-memory list of dicts."""
    if isinstance(src, str):
        if src.endswith(".jsonl"):
            raw = flatten_frames(src)
        elif src.endswith(".json"):
            raw = json.load(open(src))
        else:
            raw = pd.read_csv(src).to_dict("records")
    else:
        raw = src
    return [normalize_detection(r) for r in raw]

def load_poses(src):
    """src: path to .json/.csv, DataFrame, or list of dicts with frame_id, x, y, yaw."""
    if isinstance(src, str):
        df = pd.DataFrame(json.load(open(src))) if src.endswith(".json") else pd.read_csv(src)
    else:
        df = pd.DataFrame(src)
    return df[["frame_id", "x", "y", "yaw"]]

def stationary_poses(frame_ids):
    """Fallback if no pose is available yet: the rover stays at the origin facing +x."""
    return pd.DataFrame(dict(frame_id=sorted(set(frame_ids)), x=0.0, y=0.0, yaw=0.0))

## 3. Camera model and ground projection

* **Intrinsics** (`fx, fy, cx, cy`): the numbers that say which direction each pixel looks at. Pixel `(cx, cy)` is the image centre.
* We use the **rover frame**: `x` forward, `y` left, `z` up, origin on the ground below the camera.
* A pixel gives a **ray**. We intersect that ray with the ground plane `z = 0` (flat-ground assumption) using the camera height and pitch. That intersection is the hazard's position in the rover frame.

`ground_to_pixel` is the exact inverse, used to test the maths and to generate fake data.

In [4]:
def intrinsics(cfg=CFG):
    fx = (cfg["img_w"] / 2) / math.tan(math.radians(cfg["hfov_deg"]) / 2)
    return fx, fx, cfg["img_w"] / 2, cfg["img_h"] / 2      # fx, fy, cx, cy (square pixels)

def cam_basis(pitch_deg):
    """Camera forward / left / up axes expressed in the rover frame (camera tilted down by pitch)."""
    th = math.radians(pitch_deg)
    fwd = np.array([math.cos(th), 0.0, -math.sin(th)])
    left = np.array([0.0, 1.0, 0.0])
    up = np.array([math.sin(th), 0.0, math.cos(th)])
    return fwd, left, up

def pixel_to_ground(u, v, cfg=CFG):
    """Pixel (u, v) -> point on the ground in the rover frame [x_fwd, y_left]. None if the ray never hits the ground."""
    fx, fy, cx, cy = intrinsics(cfg)
    fwd, left, up = cam_basis(cfg["cam_pitch_deg"])
    ray = fwd + (-(u - cx) / fx) * left + (-(v - cy) / fy) * up
    if ray[2] >= -1e-6:                       # at or above the horizon
        return None
    t = cfg["cam_height_m"] / (-ray[2])       # how far along the ray until z = 0
    return np.array([t * ray[0], t * ray[1]])

def ground_to_pixel(p_r, cfg=CFG):
    """Inverse of pixel_to_ground: rover-frame ground point -> pixel (u, v)."""
    fx, fy, cx, cy = intrinsics(cfg)
    fwd, left, up = cam_basis(cfg["cam_pitch_deg"])
    vec = np.array([p_r[0], p_r[1], -cfg["cam_height_m"]])
    a, b, c = vec @ fwd, vec @ left, vec @ up
    if a <= 1e-6:
        return None
    return cx - fx * b / a, cy - fy * c / a

# sanity check: pixel -> ground -> pixel must give the same pixel back
# (use a pixel inside the 640x640 image, below the principal point so the ray hits the ground)
_p = pixel_to_ground(320, 500)
print("ground point (x fwd, y left):", np.round(_p, 3), "| round trip pixel:", np.round(ground_to_pixel(_p), 3))

ground point (x fwd, y left): [ 0.313 -0.486] | round trip pixel: [800. 500.]


## 4. Uncertainty, rover pose, grid

* **Projection error:** we shift the pixel by ±`pix_sigma` and see how far the ground point moves. Near the horizon a 1-pixel change is many metres, so far hazards automatically get a big uncertainty.
* **Pose error:** grows with the distance the rover has travelled (dead reckoning drifts).
* Both are combined into one `sigma` in metres.

In [5]:
def projection_sigma(u, v, cfg=CFG):
    p0 = pixel_to_ground(u, v, cfg)
    if p0 is None:
        return None
    s, devs = cfg["pix_sigma"], []
    for du, dv in [(s, 0), (-s, 0), (0, s), (0, -s)]:
        p = pixel_to_ground(u + du, v + dv, cfg)
        devs.append(np.linalg.norm(p - p0) if p is not None else cfg["max_range_m"])
    return max(devs)

def prepare_poses(poses_df):
    df = poses_df.sort_values("frame_id").reset_index(drop=True).copy()
    step = np.hypot(df["x"].diff().fillna(0), df["y"].diff().fillna(0))
    df["path_m"] = step.cumsum()                 # distance travelled so far
    return df.set_index("frame_id")

def rover_to_world(p_r, pose):
    c, s = math.cos(pose["yaw"]), math.sin(pose["yaw"])
    return np.array([pose["x"] + c * p_r[0] - s * p_r[1],
                     pose["y"] + s * p_r[0] + c * p_r[1]])

def to_cell(x, y, cfg=CFG):
    return int(math.floor(x / cfg["cell_size_m"])), int(math.floor(y / cfg["cell_size_m"]))

## 5. One detection → one world position

**Rocks:** we use the bottom-centre of the bounding box (where the rock touches the ground). **Craters:** a crater is a depression, so we use the centre of its box instead.
Detections above the horizon or outside `[min_range, max_range]` are rejected (and counted, so you can report why).

In [6]:
def localize_detection(det, pose, cfg=CFG):
    x1, y1, x2, y2 = det["bbox"]
    if det["cls"] == "crater":
        u, v = (x1 + x2) / 2, (y1 + y2) / 2        # crater: centre of the depression
    else:
        u, v = (x1 + x2) / 2, y2                   # rock: where it touches the ground
    p_r = pixel_to_ground(u, v, cfg)
    if p_r is None:
        return None, "above_horizon"
    rng = float(np.hypot(*p_r))
    if not (cfg["min_range_m"] <= rng <= cfg["max_range_m"]):
        return None, "out_of_range"
    sig_proj = projection_sigma(u, v, cfg)
    sig_pose = cfg["pose_sigma0_m"] + cfg["pose_drift_per_m"] * pose["path_m"]
    sigma = float(math.hypot(sig_proj, sig_pose))
    xw, yw = rover_to_world(p_r, pose)
    return dict(frame_id=det["frame_id"], timestamp=det["timestamp"], cls=det["cls"],
                conf=det["confidence"], x=float(xw), y=float(yw), sigma=sigma, range=rng), None

## 6. Merge repeated sightings (tracking)

The same rock appears in many frames. Each new observation is compared with the known hazards of the same class:
* **close enough** (`gate_m` + its uncertainty) → same hazard: merge the positions, weighting the more certain one more (inverse-variance average);
* **otherwise** → new hazard.

A hazard is **confirmed** after `min_obs` sightings (this removes one-frame false alarms).
An **event** is produced when a hazard is first confirmed, and again only if its estimate has moved by more than one grid cell or became at least twice as certain. So the link carries a handful of messages per hazard, not one per frame.

In [7]:
class Tracker:
    def __init__(self, cfg=CFG):
        self.cfg, self.tracks, self.next_id = cfg, [], 1

    def update(self, o):
        cfg, best, best_d = self.cfg, None, 1e9
        for tr in self.tracks:
            if tr["cls"] != o["cls"]:
                continue
            d = math.hypot(tr["x"] - o["x"], tr["y"] - o["y"])
            if d < cfg["gate_m"] + o["sigma"] and d < best_d:
                best, best_d = tr, d

        if best is None:                                   # new hazard
            best = dict(id=self.next_id, cls=o["cls"], x=o["x"], y=o["y"], var=o["sigma"] ** 2,
                        n=1, conf_sum=o["conf"], first_t=o["timestamp"], last_t=o["timestamp"],
                        emitted_xy=None, emitted_sigma=None)
            self.next_id += 1
            self.tracks.append(best)
        else:                                              # merge into existing hazard
            w_old, w_new = 1 / best["var"], 1 / o["sigma"] ** 2
            best["x"] = (w_old * best["x"] + w_new * o["x"]) / (w_old + w_new)
            best["y"] = (w_old * best["y"] + w_new * o["y"]) / (w_old + w_new)
            best["var"] = max(1 / (w_old + w_new), cfg["sigma_floor_m"] ** 2)
            best["n"] += 1
            best["conf_sum"] += o["conf"]
            best["last_t"] = o["timestamp"]

        if best["n"] >= cfg["min_obs"]:
            ex, sig = best["emitted_xy"], math.sqrt(best["var"])
            moved = math.inf if ex is None else math.hypot(best["x"] - ex[0], best["y"] - ex[1])
            better = ex is not None and sig <= 0.5 * best["emitted_sigma"]
            if moved >= cfg["cell_size_m"] or better:
                best["emitted_xy"], best["emitted_sigma"] = (best["x"], best["y"]), sig
                return self._event(best, o["timestamp"])
        return None

    def _event(self, tr, t):
        cx, cy = to_cell(tr["x"], tr["y"], self.cfg)
        return dict(hazard_id=tr["id"], cls=tr["cls"], x_cell=cx, y_cell=cy,
                    sigma_m=round(math.sqrt(tr["var"]), 2), timestamp=round(t, 3),
                    confidence=round(tr["conf_sum"] / tr["n"], 2), n_obs=tr["n"])

## 7. Full pipeline

In [8]:
def run_pipeline(detections, poses_df, cfg=CFG):
    poses = prepare_poses(poses_df)
    tracker, observations, events, rejects = Tracker(cfg), [], [], Counter()
    for det in sorted(detections, key=lambda d: d["frame_id"]):
        if det["confidence"] < cfg["min_conf"]:
            rejects["low_confidence"] += 1; continue
        if det["frame_id"] not in poses.index:
            rejects["no_pose_for_frame"] += 1; continue
        obs, why = localize_detection(det, poses.loc[det["frame_id"]], cfg)
        if obs is None:
            rejects[why] += 1; continue
        observations.append(obs)
        ev = tracker.update(obs)
        if ev:
            events.append(ev)
    return observations, events, tracker, rejects

## 8. Fake data for testing (skip once the real output exists)

A rover drives along a curved path past 6 known hazards. We project them into the image (with pixel noise, missed detections and false alarms) to mimic the vision output, and give the pipeline a slightly drifting pose estimate.
Because we know the true positions, we can measure the real error.

In [9]:
def make_synthetic(cfg=SYN_CFG, n_frames=160, seed=42):
    rng = np.random.default_rng(seed)
    f = np.arange(n_frames)
    tx, ty = 0.25 * f, 2.0 * np.sin(f / 35)
    tyaw = np.arctan2(np.gradient(ty), np.gradient(tx))

    truth = [("rock", 8, 3), ("rock", 15, -2.5), ("crater", 22, 2),
             ("crater", 30, -3), ("rock", 36, 1), ("ice", 12, -4)]
    dets = []
    for i in f:
        c, s = math.cos(tyaw[i]), math.sin(tyaw[i])
        for cls, hx, hy in truth:
            dx, dy = hx - tx[i], hy - ty[i]
            p_r = (c * dx + s * dy, -s * dx + c * dy)              # world -> rover frame
            px = ground_to_pixel(p_r, cfg)
            rng_m = math.hypot(*p_r)
            if px is None or not (2 <= rng_m <= 20) or rng.random() > 0.85:
                continue
            u, v = px
            w = max(15, 500 / rng_m)
            if cls == "crater":                                    # crater boxes are centred on the hazard
                box = np.array([u - w / 2, v - w / 2, u + w / 2, v + w / 2]) + rng.normal(0, 2.0, 4)
            else:                                                  # rock / ice boxes sit on the ground point
                box = np.array([u - w / 2, v - w, u + w / 2, v]) + rng.normal(0, 2.0, 4)
            if 0 <= box[0] and box[2] < cfg["img_w"] and 0 <= box[3] < cfg["img_h"]:
                dets.append(dict(frame_id=int(i), timestamp=i / 10, **{"class": cls},
                                 confidence=float(rng.uniform(0.6, 0.95)), bbox=box.tolist()))
        if rng.random() < 0.08:                                    # false alarm
            x = rng.uniform(100, 1100); y = rng.uniform(450, 700)
            dets.append(dict(frame_id=int(i), timestamp=i / 10, **{"class": str(rng.choice(["rock", "crater"]))},
                             confidence=float(rng.uniform(0.3, 0.7)), bbox=[x, y - 40, x + 40, y]))

    # estimated pose = truth + slowly drifting error
    drift = np.cumsum(rng.normal(0, 0.01, (n_frames, 2)), axis=0)
    dyaw = np.cumsum(rng.normal(0, 0.002, n_frames))
    poses = pd.DataFrame(dict(frame_id=f, x=tx + drift[:, 0], y=ty + drift[:, 1], yaw=tyaw + dyaw))
    return dets, poses, truth

## 9. Load data

In [10]:
USE_SYNTHETIC = False                 # True = run the built-in fake-data test instead
DETECTIONS_PATH = "detections.jsonl"  # teammate's detections (.jsonl, .json or .csv)
POSES_PATH = None                     # rover poses (.json or .csv); None = rover assumed stationary at origin

if USE_SYNTHETIC:
    raw_dets, poses_df, TRUTH = make_synthetic()
    detections = load_detections(raw_dets)
else:
    TRUTH = None
    detections = load_detections(DETECTIONS_PATH)
    poses_df = load_poses(POSES_PATH) if POSES_PATH else stationary_poses([d["frame_id"] for d in detections])

ACTIVE_CFG = SYN_CFG if USE_SYNTHETIC else CFG     # the fake-data test uses its own camera
print(f"{len(detections)} detections over {poses_df['frame_id'].nunique()} frames")

403 detections over 145 frames


## 10. Run

In [11]:
observations, events, tracker, rejects = run_pipeline(detections, poses_df, ACTIVE_CFG)

print(f"{len(detections)} detections -> {len(observations)} localized -> {len(tracker.tracks)} hazards "
      f"-> {len(events)} telemetry events")
print("rejected:", dict(rejects))
pd.DataFrame(events)

403 detections -> 388 localized -> 9 hazards -> 59 telemetry events
rejected: {'low_confidence': 15}


,hazard_id,cls,x_cell,y_cell,sigma_m,timestamp,confidence,n_obs
0,1,rock,4,-1,0.3,0.0,0.74,3
1,2,rock,1,-2,0.3,0.4,0.91,3
2,1,rock,3,-1,0.3,0.4,0.79,4
3,1,rock,4,-1,0.3,0.4,0.73,5
4,1,rock,3,0,0.3,0.7,0.75,6
5,1,rock,2,1,0.3,0.9,0.78,7
6,1,rock,2,0,0.3,0.9,0.81,9
7,2,rock,3,-2,0.3,0.9,0.91,4
8,1,rock,1,0,0.3,1.1,0.84,12
9,2,rock,3,-1,0.3,1.1,0.86,8


## 11. How wrong can the camera assumptions be?

The camera height, tilt and field of view are assumptions. This test changes each one a little and shows how many **metres** a few real hazard positions move. Small numbers = the result does not depend much on the guess. Large numbers = the assumption must be defended or measured.

In [12]:
tests = [("height +20%", dict(cam_height_m=ACTIVE_CFG["cam_height_m"] * 1.2)),
         ("height -20%", dict(cam_height_m=ACTIVE_CFG["cam_height_m"] * 0.8)),
         ("tilt +5 deg", dict(cam_pitch_deg=ACTIVE_CFG["cam_pitch_deg"] + 5)),
         ("tilt -5 deg", dict(cam_pitch_deg=ACTIVE_CFG["cam_pitch_deg"] - 5)),
         ("FOV +10%",    dict(hfov_deg=ACTIVE_CFG["hfov_deg"] * 1.1)),
         ("FOV -10%",    dict(hfov_deg=ACTIVE_CFG["hfov_deg"] * 0.9))]
pixels = [((d["bbox"][0] + d["bbox"][2]) / 2, d["bbox"][3]) for d in detections[:5]]   # a few real ground points

rows = []
for name, change in tests:
    cfg2 = dict(ACTIVE_CFG, **change)
    for i, p in enumerate(pixels):
        a, b = pixel_to_ground(*p, ACTIVE_CFG), pixel_to_ground(*p, cfg2)
        rows.append(dict(change=name, point=f"#{i+1}",
                         shift_m=round(float(np.hypot(*(a - b))), 2) if a is not None and b is not None else np.nan))
pd.DataFrame(rows).pivot(index="change", columns="point", values="shift_m")

point,#1,#2,#3,#4,#5
change,,,,,
FOV +10%,0.04,0.04,0.11,0.03,0.03
FOV -10%,0.03,0.04,0.09,0.03,0.03
height +20%,0.16,0.12,0.22,0.08,0.08
height -20%,0.16,0.12,0.22,0.08,0.08
tilt +5 deg,0.14,0.08,0.22,0.06,0.06
tilt -5 deg,0.18,0.10,0.32,0.07,0.06


## 12. Validation (synthetic data only)

Left: error of each single observation against its range. Far hazards should be worse, and our `sigma` should follow that trend.
Right: the map. Stars are the truth, circles the final estimates with their uncertainty.

In [13]:
if TRUTH is not None:
    T = np.array([[hx, hy] for _, hx, hy in TRUTH]); Tc = [c for c, _, _ in TRUTH]
    rows = []
    for o in observations:
        d = np.hypot(T[:, 0] - o["x"], T[:, 1] - o["y"])
        j = int(np.argmin(d))
        if d[j] < 4 and Tc[j] == o["cls"]:
            rows.append((o["range"], d[j], o["sigma"]))
    E = np.array(rows)

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].scatter(E[:, 0], E[:, 1], s=8, label="actual error")
    ax[0].scatter(E[:, 0], E[:, 2], s=8, c="orange", label="our sigma")
    ax[0].set_xlabel("range to hazard (m)"); ax[0].set_ylabel("metres"); ax[0].legend()
    ax[0].set_title("Single-frame error vs distance")

    ax[1].plot(poses_df["x"], poses_df["y"], c="gray", lw=1, label="rover path (estimated)")
    ax[1].scatter(T[:, 0], T[:, 1], marker="*", s=150, c="green", label="truth")
    for tr in tracker.tracks:
        if tr["n"] >= CFG["min_obs"]:
            ax[1].add_patch(plt.Circle((tr["x"], tr["y"]), math.sqrt(tr["var"]), fill=False, color="red"))
            ax[1].annotate(f'{tr["cls"]}#{tr["id"]}', (tr["x"], tr["y"]), fontsize=8)
    ax[1].scatter([], [], facecolors="none", edgecolors="red", label="estimate (1 sigma)")
    ax[1].set_aspect("equal"); ax[1].legend(fontsize=8); ax[1].set_title("Map (world frame, metres)")
    plt.tight_layout(); plt.show()

    print("final confirmed hazards vs truth:")
    for tr in tracker.tracks:
        if tr["n"] < CFG["min_obs"]:
            continue
        d = np.hypot(T[:, 0] - tr["x"], T[:, 1] - tr["y"]); j = int(np.argmin(d))
        print(f'  {tr["cls"]}#{tr["id"]:<2} n={tr["n"]:<3} error={d[j]:.2f} m  (claimed sigma {math.sqrt(tr["var"]):.2f} m)')
    print("unconfirmed (filtered as false alarms):", sum(t["n"] < CFG["min_obs"] for t in tracker.tracks))
else:
    print("No ground truth available with real data. Check the map visually against the video instead.")

No ground truth available with real data. Check the map visually against the video instead.


## 13. Hand-off to the telemetry step

Each event has: `hazard_id, cls, x_cell, y_cell, sigma_m, timestamp, confidence, n_obs`.
* `x_cell, y_cell` are **signed integers** (cells of `cell_size_m`, relative to the landing site).
* `sigma_m` is the uncertainty radius in metres, `timestamp` is in seconds (the dashboard uses it for data age).
* How these fields are packed into bytes is the telemetry teammate's decision.

In [14]:
with open("telemetry_events.json", "w") as fh:
    json.dump(events, fh, indent=2)
print(f"wrote {len(events)} events to telemetry_events.json")

wrote 59 events to telemetry_events.json
